# Reproducible workflow
## 30-day mortality prediction after acute myocardial infarction

This notebook is the clean, script-backed companion to `01_Complete_Executed_Analysis.ipynb`. It calls the reusable implementation in `src/mortality_prediction_pipeline.py` so the workflow can be reproduced from a notebook or from the command line.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import mortality_prediction_pipeline as pipeline

DATA = ROOT / 'ami_patient_data.csv'
OUTPUT = ROOT / 'outputs' / 'notebook_run'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Repository root:', ROOT)
print('Seed:', pipeline.RANDOM_SEED)

## 1. Data audit and cleaning
Cleaning is deterministic and auditable. The raw file is not overwritten.

In [ ]:
raw, data = pipeline.load_and_clean(DATA)
display(pipeline.audit_table(raw, data))
display(data.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame('missing_cells'))

## 2. Leakage-safe preprocessing
Median imputation + scaling are used for continuous/ordinal predictors; most-frequent imputation is used for binary predictors; smoking receives most-frequent imputation plus one-hot encoding. All trainable transformations are learned inside training folds.

In [ ]:
X = data.drop(columns=[pipeline.OUTCOME])
y = data[pipeline.OUTCOME].astype(int)
preprocessor = pipeline.make_preprocessor()
print('Shape:', X.shape, '| deaths:', int(y.sum()), '| event rate:', f'{y.mean():.2%}')

## 3. Final ridge under repeated nested cross-validation
Outer validation: 5 stratified folds × 5 repeats. Inner tuning: 4 stratified folds. Tuning criterion: log loss.

In [ ]:
models = pipeline.build_models(preprocessor)
ridge_model, ridge_grid = models['Ridge logistic']
ridge_rep, ridge_oof, ridge_tuning = pipeline.nested_oof(ridge_model, ridge_grid, X, y)
display(pd.DataFrame([pipeline.metrics(ridge_oof)]))
display(ridge_tuning.head())

## 4. Candidate-model comparison

In [ ]:
candidate_names = ['Ridge logistic', 'Standard logistic', 'Random Forest', 'Gradient Boosting']
results, oof = [], {}
for name in candidate_names:
    model, grid = models[name]
    rep, patient, tuning = pipeline.nested_oof(model, grid, X, y)
    oof[name] = patient
    results.append({'model': name, **pipeline.metrics(patient)})
_, null_patient = pipeline.null_oof(X, y)
results.append({'model': 'Intercept-only reference', **pipeline.metrics(null_patient)})
display(pd.DataFrame(results))

## 5. Class-imbalance ablation

In [ ]:
rows = []
for name in ['Ridge logistic', 'Ridge class-weighted', 'Ridge oversampled']:
    model, grid = models[name]
    _, patient, _ = pipeline.nested_oof(model, grid, X, y)
    rows.append({'strategy': name, **pipeline.metrics(patient)})
display(pd.DataFrame(rows))

## 6. Calibration and bootstrap uncertainty

In [ ]:
display(pd.DataFrame([pipeline.calibration_summary(ridge_oof)]))
display(pipeline.bootstrap_intervals(ridge_oof, n_boot=2000))

## 7. Sensitivity analyses
The reusable implementation includes categorical Killip coding and missing-indicator sensitivity analyses. See `src/mortality_prediction_pipeline.py` and the machine-readable tables under `results/tables/`.

## 8. Threshold trade-offs and decision-curve analysis

In [ ]:
import numpy as np
yy = ridge_oof['y_true'].to_numpy()
pp = ridge_oof['predicted_probability'].to_numpy()
display(pd.DataFrame([pipeline.threshold_metrics(yy, pp, t) for t in [0.05, 0.10, 0.15, 0.20]]))
dca = pipeline.decision_curve(yy, pp, np.linspace(0.01, 0.30, 60))
display(dca.head())

## 9. Final model and held-out permutation importance

In [ ]:
final_model, coefficients = pipeline.final_coefficients(X, y, preprocessor)
display(coefficients)
importance = pipeline.heldout_raw_permutation_importance(X, y, preprocessor)
display(importance)

## 10. One-command export
The line below runs the reusable pipeline and writes tables, figures, metadata and the serialized final model to `outputs/current`. It is intentionally commented because the nested workflow is computationally intensive.

In [ ]:
# pipeline.run(DATA, ROOT / 'outputs' / 'current')

## Interpretation boundary
This is an internally validated prediction model. It does not establish causal effects, treatment benefit or external transportability. External validation is required before clinical use.